## RNN 학습 루프
- 실제로 작성한 RNN 모듈을 가지고 여러 데이터에 대해서 학습 및 평가를 진행할 예정
- RNN의 장단점을 보여주는 것이 목표

### Import Packages

In [18]:
import random
import re
from typing import Dict, List, Tuple, Callable, Set
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from vanila_rnn import VanilaRNN

### `nn.Module` 정의
- 학습 대상인 RNN 언어모델 구현을 위해서 직접 구현한 `VanilaRNN` 모듈을 이용
- 입력값으로는 토큰의 임베딩 벡터를 받고, 결과값으로 각 토큰의 예측값을 반환하도록 함

![image.png](https://static.wikidocs.net/images/page/46496/rnnlm1_final_final.PNG)

In [19]:
class RNNLM(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_size: int, output_size: int, *args, **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.rnn = VanilaRNN(input_size=vocab_size, hidden_size=hidden_size)
        self.linear = nn.Linear(in_features=hidden_size, out_features=output_size)

    def forward(self, x: torch.FloatTensor, hidden = None):
        """
        Args:
            x (batch_size, seq_len, vocab_size): one-hot vector 입력 문자열
            hidden (hidden_size,): RNN의 기본 hidden vector
        """
        if x.ndim == 2:
            x = x.unsqueeze(0)
        output, hidden = self.rnn(x, hidden)
        y = self.linear(output)
        return y, hidden

### 데이터셋 구성
- RNNLM을 이용해서 텍스트의 다음 문자를 바로 예측하는 Task를 수행하고자 함
- 기본 소문자 + 공백 + 기본 문장부호 정도를 Vocabulary로 두고 진행

In [20]:
def char_tokenize(text: str) -> List[str]:
    """
    문자 기반으로 토크나이징을 수행하는 함수
    
    Args:
        text (str): 토크나이징을 수행하고자 하는 텍스트
    Returns:
        List[str]: 문자 기반으로 쪼개진 토큰들이 담긴 리스트
    """
    return list(text)

def word_tokenize(text: str) -> List[str]:
    """
    단어 기반으로 토크나이징을 수행하는 함수

    Args:
        text (str): 토크나이징을 수행하고자 하는 텍스트
    Returns:
        List[str]: 단어 기반으로 쪼개진 토큰들이 담긴 리스트
    """
    return re.findall(r"[a-zA-Z']+|[.,!?]", text)

print(f"{char_tokenize('this is an apple.')=}")
print(f"{word_tokenize('this is an apple.')=}")

char_tokenize('this is an apple.')=['t', 'h', 'i', 's', ' ', 'i', 's', ' ', 'a', 'n', ' ', 'a', 'p', 'p', 'l', 'e', '.']
word_tokenize('this is an apple.')=['this', 'is', 'an', 'apple', '.']


In [ ]:
def make_vocab(text: str, tokenize_fn: Callable) -> Tuple[Dict, Dict, Set]:
    """
    입력된 전체 코퍼스에서 토크나이징을 통해 Vocabulary를 구축하는 함수

    Args: 
        text (str): Vocabulary를 구성하고자 하는 텍스트
        tokenize_fn (Callable): 토크나이징을 수행하는 함수
    Returns:
        Dict[str, int]: (Token, Index) 쌍을 표현하는 딕셔너리
        Dict[int, str]: (Index, Token) 쌍을 표현하는 딕셔너리
    """
    tokens = sorted(set(tokenize_fn(text)))
    token2idx = {t: i for i, t in enumerate(tokens)}
    idx2token = {i: t for t, i in token2idx.items()}
    return token2idx, idx2token

chr2idx, idx2chr = make_vocab('abcdefghijklmnopqrstuvwxyz .,?!', char_tokenize)
print(chr2idx, idx2chr)

{' ': 0, '!': 1, ',': 2, '.': 3, '?': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'f': 10, 'g': 11, 'h': 12, 'i': 13, 'j': 14, 'k': 15, 'l': 16, 'm': 17, 'n': 18, 'o': 19, 'p': 20, 'q': 21, 'r': 22, 's': 23, 't': 24, 'u': 25, 'v': 26, 'w': 27, 'x': 28, 'y': 29, 'z': 30} {0: ' ', 1: '!', 2: ',', 3: '.', 4: '?', 5: 'a', 6: 'b', 7: 'c', 8: 'd', 9: 'e', 10: 'f', 11: 'g', 12: 'h', 13: 'i', 14: 'j', 15: 'k', 16: 'l', 17: 'm', 18: 'n', 19: 'o', 20: 'p', 21: 'q', 22: 'r', 23: 's', 24: 't', 25: 'u', 26: 'v', 27: 'w', 28: 'x', 29: 'y', 30: 'z'} [' ', '!', ',', '.', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
def encoder(tokens: list, vocab: Dict[str, int]):
    token_ids = [vocab[token] for token in tokens]
    return token_ids

def decoder(token_ids: list, vocab: Dict[int, str]):
    tokens = [vocab[token_id] for token_id in token_ids]
    return tokens
    
print(encoder(list("i love you"), chr2idx))
print(decoder([13, 0, 16, 19, 26, 9, 0, 29, 19, 25], idx2chr))

[13, 0, 16, 19, 26, 9, 0, 29, 19, 25]
['i', ' ', 'l', 'o', 'v', 'e', ' ', 'y', 'o', 'u']


In [5]:
def make_repeating_data(pattern: str, length: int):
    text = pattern * (length // len(pattern)) + pattern[:(length % len(pattern))]
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

def make_long_range_data(char: str, N: int, vocab: list):
    vocab_available = vocab[:]
    vocab_available.remove(char)
    padding = [random.choice(vocab_available) for _ in range(N)]

    text = char + ''.join(padding) + char
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

def make_corpus_data(text: str):
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

### 학습 파이프라인 구성

In [6]:
def train(model: nn.Module, inputs, labels, loss_fn, optimizer, device: str = "cpu"):
    # 1. model을 학습 모드로 설정
    model.train()
    running_loss = 0.0
    
    for X, y in zip(inputs, labels):
        X, y = X.to(device), y.to(device)

        # 2. 이전 step에서의 grad 초기화
        optimizer.zero_grad()

        # 3. Loss 계산 (Forward)
        outputs, hidden = model.forward(X)
        loss = loss_fn(outputs.reshape(-1, vocab_size), y.reshape(-1))

        # 4. Gradient 계산 (Backward)
        loss.backward()

        # 5. Weight 업데이트 (Optimization)
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(inputs)

In [7]:
# 1. 텍스트 반복 데이터셋 생성
patterns = ["i love you ", "hi", "this is an apple. ", "the weather is hot. ", "wow!"]
repeating_dataset = [make_repeating_data(pattern, 20) for pattern in patterns]
inputs, outputs = zip(*repeating_dataset)
inputs = F.one_hot(torch.LongTensor(inputs), num_classes=vocab_size).float()
outputs = torch.LongTensor(outputs)

In [8]:
model = RNNLM(vocab_size=vocab_size, hidden_size=128, output_size=vocab_size)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
num_epochs = 100

for epoch in tqdm(range(num_epochs)):
    train_loss = train(model, inputs, outputs, loss_fn, optimizer)
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}")

  9%|▉         | 9/100 [00:00<00:01, 47.82it/s]

Epoch 01 | Train Loss: 3.5616
Epoch 02 | Train Loss: 3.5477
Epoch 03 | Train Loss: 3.5640
Epoch 04 | Train Loss: 3.4636
Epoch 05 | Train Loss: 3.4532
Epoch 06 | Train Loss: 3.2956
Epoch 07 | Train Loss: 3.2415
Epoch 08 | Train Loss: 3.1623
Epoch 09 | Train Loss: 3.1777
Epoch 10 | Train Loss: 3.1446
Epoch 11 | Train Loss: 3.0206
Epoch 12 | Train Loss: 3.0043
Epoch 13 | Train Loss: 2.8540
Epoch 14 | Train Loss: 2.9767
Epoch 15 | Train Loss: 2.7721
Epoch 16 | Train Loss: 2.8644
Epoch 17 | Train Loss: 2.7919
Epoch 18 | Train Loss: 2.7190
Epoch 19 | Train Loss: 2.7514


 30%|███       | 30/100 [00:00<00:00, 83.11it/s]

Epoch 20 | Train Loss: 2.6993
Epoch 21 | Train Loss: 2.6419
Epoch 22 | Train Loss: 2.7056
Epoch 23 | Train Loss: 2.5823
Epoch 24 | Train Loss: 2.5700
Epoch 25 | Train Loss: 2.3720
Epoch 26 | Train Loss: 2.5832
Epoch 27 | Train Loss: 2.3780
Epoch 28 | Train Loss: 2.3850
Epoch 29 | Train Loss: 2.4443
Epoch 30 | Train Loss: 2.2158
Epoch 31 | Train Loss: 2.2708
Epoch 32 | Train Loss: 2.3936
Epoch 33 | Train Loss: 2.5354
Epoch 34 | Train Loss: 2.2609
Epoch 35 | Train Loss: 2.2255
Epoch 36 | Train Loss: 2.1803
Epoch 37 | Train Loss: 2.2401
Epoch 38 | Train Loss: 2.4938
Epoch 39 | Train Loss: 2.3235
Epoch 40 | Train Loss: 2.2573


 52%|█████▏    | 52/100 [00:00<00:00, 94.85it/s]

Epoch 41 | Train Loss: 2.3688
Epoch 42 | Train Loss: 2.1862
Epoch 43 | Train Loss: 2.2034
Epoch 44 | Train Loss: 2.2952
Epoch 45 | Train Loss: 2.2460
Epoch 46 | Train Loss: 2.1499
Epoch 47 | Train Loss: 2.2631
Epoch 48 | Train Loss: 2.2935
Epoch 49 | Train Loss: 2.2278
Epoch 50 | Train Loss: 2.0492
Epoch 51 | Train Loss: 2.0453
Epoch 52 | Train Loss: 1.8939
Epoch 53 | Train Loss: 2.0000
Epoch 54 | Train Loss: 1.9918
Epoch 55 | Train Loss: 2.0909
Epoch 56 | Train Loss: 2.0389
Epoch 57 | Train Loss: 2.2042
Epoch 58 | Train Loss: 2.1621
Epoch 59 | Train Loss: 2.2944
Epoch 60 | Train Loss: 2.0233
Epoch 61 | Train Loss: 1.9793
Epoch 62 | Train Loss: 1.8212


 73%|███████▎  | 73/100 [00:00<00:00, 92.04it/s]

Epoch 63 | Train Loss: 1.7626
Epoch 64 | Train Loss: 1.9552
Epoch 65 | Train Loss: 1.7773
Epoch 66 | Train Loss: 1.9669
Epoch 67 | Train Loss: 1.9012
Epoch 68 | Train Loss: 1.7779
Epoch 69 | Train Loss: 1.7770
Epoch 70 | Train Loss: 1.9303
Epoch 71 | Train Loss: 2.0917
Epoch 72 | Train Loss: 1.8607
Epoch 73 | Train Loss: 1.8373
Epoch 74 | Train Loss: 1.8334
Epoch 75 | Train Loss: 1.9043
Epoch 76 | Train Loss: 1.8377
Epoch 77 | Train Loss: 1.8638
Epoch 78 | Train Loss: 1.8612
Epoch 79 | Train Loss: 2.0806
Epoch 80 | Train Loss: 1.9266


100%|██████████| 100/100 [00:01<00:00, 88.04it/s]

Epoch 81 | Train Loss: 1.9412
Epoch 82 | Train Loss: 1.7507
Epoch 83 | Train Loss: 1.7984
Epoch 84 | Train Loss: 1.8167
Epoch 85 | Train Loss: 1.7792
Epoch 86 | Train Loss: 1.7820
Epoch 87 | Train Loss: 1.6907
Epoch 88 | Train Loss: 1.8430
Epoch 89 | Train Loss: 1.7011
Epoch 90 | Train Loss: 1.7132
Epoch 91 | Train Loss: 1.7497
Epoch 92 | Train Loss: 1.7711
Epoch 93 | Train Loss: 1.7085
Epoch 94 | Train Loss: 1.7304
Epoch 95 | Train Loss: 1.8014
Epoch 96 | Train Loss: 1.7799
Epoch 97 | Train Loss: 1.7126
Epoch 98 | Train Loss: 1.7754
Epoch 99 | Train Loss: 1.6711
Epoch 100 | Train Loss: 1.6483
